In [9]:

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer, util
import os
from datetime import datetime

c:\Users\atorr\miniconda3\envs\acomparto\Lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
c:\Users\atorr\miniconda3\envs\acomparto\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [14]:
df_responses = pd.read_excel("expert_responses.xlsx",)

In [15]:
df_negative_responses = pd.read_excel("expert_negative_responses.xlsx",)

In [17]:
df_chat =  pd.read_excel("chat_responses.xlsx",header=None)

In [18]:
MODEL_NAME = "paraphrase-multilingual-mpnet-base-v2"  # buen modelo multilingüe para español
OUTFILE = "bert_similarity_results_negative.txt"

In [19]:
model = SentenceTransformer(MODEL_NAME)

In [20]:
question = "¿puedo tomar alcohol durante el embarazo?"

In [21]:
human_positive = [
    "No se recomienda tomar alcohol durante el embarazo, ya que puede causar daños al feto y al desarrollo prenatal.",
    "Lo más seguro es evitar completamente el alcohol durante la gestación porque incluso pequeñas cantidades pueden implicar riesgos.",
    "Los profesionales de salud aconsejan abstenerse de alcohol mientras se está embarazada para prevenir problemas de desarrollo."
]

# 15 ejemplos NEGATIVOS (parrafos largos, irrelevantes — como pediste)
human_negative = [
    "La historia de la aviación moderna comienza con los hermanos Wright, quienes lograron el primer vuelo controlado en 1903. Desde entonces, la tecnología aeronáutica ha evolucionado a pasos agigantados, permitiendo vuelos supersónicos y exploración espacial. Hoy en día, los aviones no solo transportan pasajeros, sino que también desempeñan un papel clave en la logística global.",
    "El proceso de fotosíntesis en las plantas consiste en la conversión de energía lumínica en energía química. A través de la clorofila, las plantas capturan la luz solar y transforman dióxido de carbono y agua en glucosa y oxígeno. Este mecanismo es esencial para la vida en la Tierra, ya que mantiene el equilibrio de los ecosistemas y la producción de oxígeno.",
    "En la Edad Media, los castillos eran el centro de poder militar y político en Europa. Estas fortalezas estaban construidas con gruesos muros de piedra, torres de vigilancia y fosos para proteger a sus habitantes de posibles invasiones. Además, los castillos no solo servían como defensa, sino también como símbolo de prestigio y control territorial.",
    "El cambio climático es uno de los desafíos más grandes de nuestra época. El aumento de la temperatura global está provocando el derretimiento de los glaciares, la subida del nivel del mar y fenómenos meteorológicos extremos. Para enfrentar este problema, los países deben cooperar en la reducción de emisiones de gases de efecto invernadero y en la adopción de energías renovables.",
    "La programación orientada a objetos es un paradigma de desarrollo de software que organiza el código en torno a objetos y clases. Este enfoque permite crear sistemas más modulares, reutilizables y fáciles de mantener. Lenguajes como Java, Python y C++ se basan en este paradigma para resolver problemas complejos en el ámbito tecnológico.",
    "El ajedrez es un juego de estrategia que se ha practicado durante siglos. Requiere concentración, paciencia y habilidades de planificación a largo plazo. Los grandes maestros del ajedrez son capaces de anticipar varios movimientos de su oponente y diseñar jugadas que aseguren el control del tablero y la victoria final.",
    "La música clásica ha tenido una gran influencia en la cultura occidental. Compositores como Mozart, Beethoven y Bach crearon obras maestras que todavía se interpretan en teatros y salas de concierto en todo el mundo. Además, la música clásica ha servido como inspiración para otros géneros musicales modernos.",
    "La historia del cine comenzó con los hermanos Lumière en 1895, cuando presentaron al público las primeras proyecciones de películas. Desde entonces, la industria cinematográfica se ha convertido en un medio de entretenimiento masivo que abarca desde películas mudas hasta producciones en 3D y plataformas de streaming.",
    "El cuerpo humano está compuesto por diversos sistemas que trabajan en conjunto para mantener la vida. Entre ellos, el sistema nervioso controla las funciones corporales y permite la comunicación entre los órganos. El sistema circulatorio distribuye oxígeno y nutrientes a través de la sangre, mientras que el sistema digestivo transforma los alimentos en energía.",
    "Los océanos cubren más del 70% de la superficie de la Tierra y son el hogar de millones de especies marinas. Sin embargo, la contaminación plástica y la sobrepesca están poniendo en peligro la biodiversidad marina. Es urgente implementar políticas internacionales para proteger los ecosistemas marinos y garantizar la sostenibilidad de los recursos acuáticos.",
    "En el ámbito deportivo, el fútbol es considerado el deporte más popular del mundo. Cada cuatro años, la Copa Mundial de la FIFA reúne a las mejores selecciones nacionales y capta la atención de millones de espectadores. Más allá de la competencia, el fútbol fomenta la unión cultural y la pasión entre los aficionados.",
    "La inteligencia artificial está revolucionando múltiples sectores de la sociedad. Desde los asistentes virtuales en dispositivos móviles hasta los sistemas de diagnóstico médico, la IA permite optimizar procesos y tomar decisiones más rápidas y eficientes. Sin embargo, también plantea retos éticos relacionados con la privacidad y el uso responsable de los datos.",
    "El arte del Renacimiento marcó un antes y un después en la historia cultural de Europa. Artistas como Leonardo da Vinci y Miguel Ángel introdujeron nuevas técnicas de perspectiva y anatomía que transformaron la forma de representar la realidad. Sus obras siguen siendo un referente estético y un patrimonio invaluable de la humanidad.",
    "Las lenguas del mundo presentan una diversidad fascinante. Algunas, como el chino mandarín, son habladas por millones de personas, mientras que otras lenguas minoritarias corren el riesgo de desaparecer. La preservación de estas lenguas es importante para conservar la identidad cultural de las comunidades que las utilizan.",
    "La exploración espacial ha permitido al ser humano ampliar su conocimiento del universo. Desde la llegada del hombre a la Luna en 1969 hasta las misiones actuales a Marte, los avances tecnológicos han sido impresionantes. Estos proyectos no solo buscan responder preguntas científicas, sino también abrir posibilidades para la colonización de otros planetas."
]


In [22]:
# Dos respuestas de chat a probar (una correcta / una incorrecta)
chat_good = "No, se recomienda evitar el consumo de alcohol durante el embarazo porque puede causar malformaciones y problemas en el desarrollo del feto."
chat_bad  = "Sí, tomar una copa ocasional no tiene importancia; no hay evidencia consistente de daño si es poco."


In [23]:
def compute_similarity_stats(model, hypothesis, references):
    """
    Devuelve una serie con todas las similitudes individuales y stats:
    mean, std, median.
    """
    # codificar: devuelve tensores
    emb_hyp = model.encode(hypothesis, convert_to_tensor=True)
    emb_refs = model.encode(references, convert_to_tensor=True)

    # matriz de similitud (refs x hyp)
    sims = util.cos_sim(emb_refs, emb_hyp)  # tensor (n_refs, 1)
    sims = sims.squeeze(dim=1).cpu().numpy()  # ahora (n_refs,)

    return sims, sims.mean(), sims.std(ddof=0), np.median(sims)


In [20]:
len(df_responses.iloc[0])

14

In [24]:
with open("results_bert_experts_negative.txt", "w", encoding="utf-8") as f:
    

    for i in range(1):
        #sim_per_question = []
        f.write(f"Pregunta {i}\n")
        
        similitud = 0
        rows = []
        #sentences =  [ref for ref in df_responses.iloc[:,i].tolist()]
        #sentences = [s for s in sentences if len(s.split()) > 4]
        chat_sentence =[df_chat.iloc[i,1]]
        #human_negative
        for question in (human_negative):
            sims_pos, mean_pos, std_pos, med_pos = compute_similarity_stats(model, chat_sentence, question)
            similitud+=sims_pos
            #sim = cosine_similarity(chat_sentence_embed, sentence_embed)[0][0]
            f.write("=== Chat (correcto) vs Respuestas HUMANAS (positivas) ===\n")
            for i, s in enumerate(sims_pos, 1):
                f.write(f"Pos {i}: {s:.4f}\n")
            f.write(f"Mean: {mean_pos:.4f}   Std: {std_pos:.4f}   Median: {med_pos:.4f}\n\n")
            rows.append(["chat", "positives", mean_pos, std_pos, med_pos])

            #print(similitud)
        res = similitud/len(human_negative)
        f.write(f"Promedio {res}n")
        #score_per_question.append(res)
        #print(f"similarity score con BERT: {res:.4f}")

In [26]:
chat_sentence

[' Gracias por tu pregunta. Es importante tener en cuenta que la ingestión de alcohol puede tener efectos negativos sobre el crianza del bebé. Aunque la cantidad de alcohol en la leche materna disminuye con el tiempo, es recomendable evitar consumir bebidas alcohólicas durante la lactancia. Es recomendable hablar con un especialista en salud si tienes dudas o preguntas adicionales sobre esta temática.']

In [25]:
res

array([0.09575654], dtype=float32)

In [16]:
sentences

['¿Podrías escribir la respuesta corta que tú dirías, considerando elementos que se mencionan en las instrucciones?',
 '¿Podrías escribir la respuesta corta que tú dirías, considerando elementos que se mencionan en las instrucciones?2',
 '¿Podrías escribir la respuesta corta que tú dirías, considerando elementos que se mencionan en las instrucciones?3',
 '¿Podrías escribir la respuesta corta que tú dirías, considerando elementos que se mencionan en las instrucciones?4',
 '¿Podrías escribir la respuesta corta que tú dirías, considerando elementos que se mencionan en las instrucciones?5',
 '¿Podrías escribir la respuesta corta que tú dirías, considerando elementos que se mencionan en las instrucciones?6',
 '¿Podrías escribir la respuesta corta que tú dirías, considerando elementos que se mencionan en las instrucciones?7',
 '¿Podrías escribir la respuesta corta que tú dirías, considerando elementos que se mencionan en las instrucciones?8',
 '¿Podrías escribir la respuesta corta que tú dir

In [17]:
print(model)
print( chat_good)
print( human_positive)

SentenceTransformer(
  (0): Transformer({'max_seq_length': 128, 'do_lower_case': False, 'architecture': 'XLMRobertaModel'})
  (1): Pooling({'word_embedding_dimension': 768, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
)
No, se recomienda evitar el consumo de alcohol durante el embarazo porque puede causar malformaciones y problemas en el desarrollo del feto.
['No se recomienda tomar alcohol durante el embarazo, ya que puede causar daños al feto y al desarrollo prenatal.', 'Lo más seguro es evitar completamente el alcohol durante la gestación porque incluso pequeñas cantidades pueden implicar riesgos.', 'Los profesionales de salud aconsejan abstenerse de alcohol mientras se está embarazada para prevenir problemas de desarrollo.']


In [ ]:
def run_test_and_save(outfile=OUTFILE):
    rows = []
    now = datetime.now().isoformat(timespec='seconds')
    header = f"Reporte de similitud BERT - modelo {MODEL_NAME}\nFecha: {now}\nPregunta: {question}\n\n"

    with open(outfile, "w", encoding="utf-8") as f:
        f.write(header)

        # Prueba 1: chat_good vs positivos
        sims_pos, mean_pos, std_pos, med_pos = compute_similarity_stats(model, chat_good, human_positive)
        f.write("=== Chat (correcto) vs Respuestas HUMANAS (positivas) ===\n")
        for i, s in enumerate(sims_pos, 1):
            f.write(f"Pos {i}: {s:.4f}\n")
        f.write(f"Mean: {mean_pos:.4f}   Std: {std_pos:.4f}   Median: {med_pos:.4f}\n\n")
        rows.append(["chat_good", "positives", mean_pos, std_pos, med_pos])

        # Prueba 2: chat_good vs negativos
        sims_neg, mean_neg, std_neg, med_neg = compute_similarity_stats(model, chat_good, human_negative)
        f.write("=== Chat (correcto) vs Ejemplos NEGATIVOS ===\n")
        for i, s in enumerate(sims_neg, 1):
            f.write(f"Neg {i}: {s:.4f}\n")
        f.write(f"Mean: {mean_neg:.4f}   Std: {std_neg:.4f}   Median: {med_neg:.4f}\n\n")
        rows.append(["chat_good", "negatives", mean_neg, std_neg, med_neg])

        
        # resumen en tabla
        df_summary = pd.DataFrame(rows, columns=["chat_response","set","mean","std","median"])
        f.write("=== RESUMEN TABULAR ===\n")
        f.write(df_summary.to_string(index=False))
        f.write("\n")

    print(f"Resultados guardados en: {os.path.abspath(outfile)}")
    return df_summary



In [9]:
if __name__ == "__main__":
    summary_df = run_test_and_save()
    print("\nResumen:")
    print(summary_df.to_string(index=False))

Resultados guardados en: c:\Users\atorr\OneDrive - Instituto Tecnologico y de Estudios Superiores de Monterrey\Documents\Python Proyects\Acomparto\Acomparto\bert_similarity_results.txt

Resumen:
chat_response       set     mean      std   median
    chat_good positives 0.920087 0.037386 0.909932
    chat_good negatives 0.076668 0.086023 0.044601
     chat_bad positives 0.359941 0.025741 0.357866
     chat_bad negatives 0.068294 0.071759 0.070937
